# Scaling the web classifier — a proper training harness

Step one of pushing toward practical use: stop training on one or two realizations with a hand-rolled loop, and build a real harness on all 27. This notebook adds the pieces that matter for scaling to raw datasets:

- **Train / validation / test split across realizations** — a genuinely held-out test set, so the reported numbers mean something.
- **An on-the-fly data pipeline** (`Dataset` + `DataLoader`) — labels and tracer fields are computed once per realization, then sub-cubes are served and augmented on demand, instead of materializing one giant tensor. This is the pattern that scales to bigger grids and more sims.
- **Physically-valid augmentation** — the field is periodic and the web labels are invariant under the cube's rotations and reflections, so random flips + axis permutations multiply the effective data ~48× for free. This is the cheapest lever on the rare knot class.
- **GPU + mixed precision, cosine LR schedule, best-checkpoint** — standard training hygiene that the toy loop skipped.
- **Honest metrics** — per-class F1, mean IoU, and a confusion matrix, all on the held-out test set.

The defaults assume a GPU; on CPU, shrink the ID lists and epochs. Auto-detects CUDA.

## 0. Setup, data, config

In [ ]:
# torch already in env
!pip install torch

In [ ]:
import os, time, copy, urllib.request, numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from numpy.fft import fftn, ifftn, fftfreq
torch.manual_seed(0); np.random.seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_amp = (device == 'cuda')

URL   = "https://users.flatironinstitute.org/~camels/CMD/3D_grids/data/Nbody/Grids_Mtot_Nbody_IllustrisTNG_CV_128_z=0.0.npy"
FNAME = "cmd_cv128.npy"

# ---- config (defaults sized for a GPU; shrink ID lists + epochs for CPU) ----
L, R_smooth, lam_th, nbar = 25.0, 1.0, 0.3, 3.0
patch, stride   = 32, 16        # stride < patch => overlapping cubes => more training data
TRAIN_IDS = list(range(0,19))   # 19 realizations to train
VAL_IDS   = [19,20,21,22]       # 4 to validate / pick best epoch
TEST_IDS  = [23,24,25,26]       # 4 held out for the final report
f3d, epochs, batch, lr = 16, 40, 8, 1e-3
AUGMENT, NUM_WORKERS = True, 0   # bump NUM_WORKERS (e.g. 4) on a GPU box to feed the card
CLASS_NAMES  = ['void','sheet','filament','knot']
CLASS_COLORS = ['#08306b','#6baed6','#fd8d3c','#cb181d']
CMAP=ListedColormap(CLASS_COLORS); NORM=BoundaryNorm([-.5,.5,1.5,2.5,3.5],4)

if not os.path.exists(FNAME):
    print("downloading ~216 MB ..."); urllib.request.urlretrieve(URL, FNAME)
print("device:", device, "| AMP:", use_amp, "| data:", FNAME)

## 1. Precompute labels and tracer fields (once per realization)

The T-web is the expensive part, so we run it once for each realization we'll use and cache the result in memory. Inputs are standardized with statistics fit on the **training** realizations only.

In [ ]:
def smooth(d,L,R):
    N=d.shape[0]; kax=fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    return ifftn(fftn(d)*np.exp(-0.5*(KX**2+KY**2+KZ**2)*R**2)).real
def tweb(d,L,lam):
    N=d.shape[0]; kax=fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    Kv=[KX,KY,KZ]; k2=KX**2+KY**2+KZ**2; k2[0,0,0]=1.0; dk=fftn(d); T=np.zeros((N,N,N,3,3))
    for i in range(3):
        for j in range(i,3):
            t=ifftn((Kv[i]*Kv[j]/k2)*dk).real; T[...,i,j]=t
            if i!=j: T[...,j,i]=t
    return np.sum(np.linalg.eigvalsh(T)>lam,-1).astype(np.int8)
def prep(rho, seed):
    delta = rho/rho.mean()-1.0
    lab = tweb(smooth(delta,L,R_smooth),L,lam_th)
    counts = np.random.default_rng(seed).poisson(nbar*np.clip(1+delta,0,None))
    return np.log10(1.0+counts).astype(np.float32), lab

grids = np.load(FNAME, mmap_mode='r')
all_ids = sorted(set(TRAIN_IDS)|set(VAL_IDS)|set(TEST_IDS))
INP, LAB = {}, {}
t0=time.time()
for n,gi in enumerate(all_ids):
    INP[gi], LAB[gi] = prep(np.array(grids[gi],dtype=np.float64), 100+gi)
    print(f"\r  prepped {n+1}/{len(all_ids)} realizations ({time.time()-t0:.0f}s)", end="")
mu = np.mean([INP[gi].mean() for gi in TRAIN_IDS]); sd = np.mean([INP[gi].std() for gi in TRAIN_IDS])
for gi in all_ids: INP[gi] = (INP[gi]-mu)/sd     # standardize with train stats
fr = np.bincount(np.concatenate([LAB[gi].ravel() for gi in TRAIN_IDS]), minlength=4)
print(f"\nclass mix (train) %: {dict(zip(CLASS_NAMES,(100*fr/fr.sum()).round(2)))}")

## 2. Dataset, augmentation, and loaders

`WebSet` indexes every sub-cube corner across its realizations and serves them on demand. With `augment=True` each cube gets a random element of the cube symmetry group (axis flips + permutation) applied to input and label together — valid because the web labels are rotation/reflection invariant.

In [ ]:
def cube_symmetry(x, y):
    for ax in range(3):
        if np.random.random() < 0.5:
            x = np.flip(x, ax); y = np.flip(y, ax)
    perm = np.random.permutation(3)
    x = np.transpose(x, perm); y = np.transpose(y, perm)
    return np.ascontiguousarray(x), np.ascontiguousarray(y)

class WebSet(Dataset):
    def __init__(self, ids, augment=False):
        self.augment=augment; self.index=[]
        N=next(iter(INP.values())).shape[0]
        for gi in ids:
            for i in range(0,N-patch+1,stride):
                for j in range(0,N-patch+1,stride):
                    for k in range(0,N-patch+1,stride):
                        self.index.append((gi,i,j,k))
    def __len__(self): return len(self.index)
    def __getitem__(self, n):
        gi,i,j,k = self.index[n]
        x = INP[gi][i:i+patch, j:j+patch, k:k+patch]
        y = LAB[gi][i:i+patch, j:j+patch, k:k+patch].astype(np.int64)
        if self.augment: x,y = cube_symmetry(x,y)
        return torch.from_numpy(x.copy())[None].float(), torch.from_numpy(y.copy())

train_dl = DataLoader(WebSet(TRAIN_IDS, AUGMENT), batch_size=batch, shuffle=True,  num_workers=NUM_WORKERS, drop_last=True)
val_dl   = DataLoader(WebSet(VAL_IDS),            batch_size=batch, shuffle=False, num_workers=NUM_WORKERS)
test_dl  = DataLoader(WebSet(TEST_IDS),           batch_size=batch, shuffle=False, num_workers=NUM_WORKERS)
print(f"cubes  train {len(train_dl.dataset)}  val {len(val_dl.dataset)}  test {len(test_dl.dataset)}")

## 3. The 3D U-Net and training loop

Cross-entropy with gentle (sqrt-inverse-frequency) class weights, cosine-annealed LR, mixed precision on GPU, and we keep the weights from the epoch with the best validation macro-F1.

In [ ]:
def double_conv(ci,co):
    return nn.Sequential(nn.Conv3d(ci,co,3,padding=1),nn.BatchNorm3d(co),nn.ReLU(True),
                         nn.Conv3d(co,co,3,padding=1),nn.BatchNorm3d(co),nn.ReLU(True))
class UNet3D(nn.Module):
    def __init__(self,f=16,nc=4):
        super().__init__(); self.e1=double_conv(1,f); self.e2=double_conv(f,2*f); self.b=double_conv(2*f,4*f)
        self.u2=nn.ConvTranspose3d(4*f,2*f,2,2); self.d2=double_conv(4*f,2*f)
        self.u1=nn.ConvTranspose3d(2*f,f,2,2); self.d1=double_conv(2*f,f); self.o=nn.Conv3d(f,nc,1); self.p=nn.MaxPool3d(2)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.p(e1)); b=self.b(self.p(e2))
        d2=self.d2(torch.cat([self.u2(b),e2],1)); d1=self.d1(torch.cat([self.u1(d2),e1],1)); return self.o(d1)

def confusion(net, dl, nc=4):
    net.eval(); C=np.zeros((nc,nc),dtype=np.int64)
    with torch.no_grad():
        for x,y in dl:
            p=net(x.to(device)).argmax(1).cpu().numpy().ravel(); t=y.numpy().ravel()
            C += np.bincount(nc*t+p, minlength=nc*nc).reshape(nc,nc)
    return C
def metrics(C):
    f1,iou=[],[]
    for c in range(len(C)):
        tp=C[c,c]; fp=C[:,c].sum()-tp; fn=C[c,:].sum()-tp
        f1.append(2*tp/(2*tp+fp+fn+1e-9)); iou.append(tp/(tp+fp+fn+1e-9))
    acc=np.trace(C)/C.sum()
    return acc, np.array(f1), np.array(iou)

w = 1.0/np.sqrt(fr+1); w = (w/w.sum()*4).astype(np.float32)
net = UNet3D(f3d).to(device)
opt = torch.optim.Adam(net.parameters(), lr)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
crit = nn.CrossEntropyLoss(weight=torch.tensor(w).to(device))
scaler = torch.amp.GradScaler(device, enabled=use_amp)
print(f"{sum(p.numel() for p in net.parameters())} params on {device}")

best_f1, best_state = 0.0, None; t0=time.time()
for ep in range(epochs):
    net.train()
    for x,y in train_dl:
        x,y = x.to(device), y.to(device); opt.zero_grad()
        with torch.amp.autocast(device_type=device, enabled=use_amp):
            loss = crit(net(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    sched.step()
    acc,f1,iou = metrics(confusion(net,val_dl))
    if f1.mean() > best_f1: best_f1, best_state = f1.mean(), copy.deepcopy(net.state_dict())
    if ep%5==4 or ep==epochs-1 or ep<2:
        print(f"epoch {ep+1:2d}  valAcc {acc:.3f}  macroF1 {f1.mean():.3f}  "
              f"(V {f1[0]:.2f} S {f1[1]:.2f} F {f1[2]:.2f} K {f1[3]:.2f})  ({time.time()-t0:.0f}s)")
net.load_state_dict(best_state)
print(f"\nloaded best model (val macroF1 {best_f1:.3f})")

## 4. Held-out test report

The numbers that count — computed once, on realizations the model never saw during training or selection.

In [ ]:
C = confusion(net, test_dl)
acc, f1, iou = metrics(C)
print(f"TEST  accuracy {acc:.3f}   mean IoU {iou.mean():.3f}   macro F1 {f1.mean():.3f}")
for c,name in enumerate(CLASS_NAMES):
    print(f"  {name:9s}  F1 {f1[c]:.3f}   IoU {iou[c]:.3f}")

Cn = C/C.sum(1,keepdims=True)
fig,ax=plt.subplots(figsize=(5.5,5))
im=ax.imshow(Cn,cmap='Blues',vmin=0,vmax=1)
ax.set_xticks(range(4)); ax.set_yticks(range(4)); ax.set_xticklabels(CLASS_NAMES); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title('Test confusion (row-normalized)')
for i in range(4):
    for j in range(4):
        ax.text(j,i,f"{Cn[i,j]:.2f}",ha='center',va='center',color='white' if Cn[i,j]>0.5 else 'black')
plt.colorbar(im,fraction=0.046); plt.tight_layout(); plt.show()

## What this gives you, and the next lever

You now have a scalable training harness: held-out evaluation, an augmented on-the-fly data pipeline, and GPU-ready training. The knobs that push performance from here, in order of leverage:

- **More cubes** — shrink `stride` (more overlap) and add realizations to `TRAIN_IDS`. Augmentation + more data is what lifts the knot class.
- **A wider/deeper net** — raise `f3d`, or add a third down/up level, once the GPU has headroom.
- **Longer training** — more `epochs`; the cosine schedule already anneals the LR for you.

With the harness solid, the other moves slot in cleanly: the **VIDE / DisPerSE void cross-check** (does the network's "void" agree with the catalog finders?), and **cosmology variation** — swap the CV file for the LH set's 1,000 universes, attach each grid's (Ωm, σ8) labels, and the head becomes a *parameter-inference* network instead of a segmenter.